In [ ]:
"""
TransE Model Training with Optuna & Customized Analysis (AdamW Version)

Features:
1. Optimizer changed to AdamW.
2. Automated Output Management (Saves to ./output/ folder).
"""

from __future__ import absolute_import, division, print_function

import random
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import optuna
from optuna.trial import TrialState
import matplotlib.pyplot as plt
import warnings
import time

warnings.filterwarnings("ignore")

# Set random seeds
SEED = 2025
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True


class PyTorchTrainDataset(Dataset):
    """Dataset class for TransE training with negative sampling."""

    def __init__(
        self,
        head,
        tail,
        rel,
        ent_total,
        rel_total,
        sampling_mode="normal",
        bern_flag=False,
        filter_flag=True,
        neg_ent=1,
        neg_rel=0,
    ):
        self.head = head
        self.tail = tail
        self.rel = rel
        self.rel_total = rel_total
        self.ent_total = ent_total
        self.tri_total = len(head)
        self.sampling_mode = sampling_mode
        self.neg_ent = neg_ent
        self.neg_rel = neg_rel
        self.bern_flag = bern_flag
        self.filter_flag = filter_flag

        print("Building mapping dictionaries (this may take a moment)...")
        self.__count_htr()
        print("Mapping dictionaries built.")

    def __len__(self):
        return self.tri_total

    def __getitem__(self, idx):
        return (self.head[idx], self.tail[idx], self.rel[idx])

    def collate_fn(self, data):
        batch_data = {}
        batch_h = np.array([item[0] for item in data]).reshape(-1, 1)
        batch_t = np.array([item[1] for item in data]).reshape(-1, 1)
        batch_r = np.array([item[2] for item in data]).reshape(-1, 1)
        batch_h = np.repeat(batch_h, 1 + self.neg_ent + self.neg_rel, axis=-1)
        batch_t = np.repeat(batch_t, 1 + self.neg_ent + self.neg_rel, axis=-1)
        batch_r = np.repeat(batch_r, 1 + self.neg_ent + self.neg_rel, axis=-1)

        for index, item in enumerate(data):
            last = 1
            if self.neg_ent > 0:
                neg_head, neg_tail = self.__normal_batch(
                    item[0], item[1], item[2], self.neg_ent
                )
                if len(neg_head) > 0:
                    batch_h[index][last : last + len(neg_head)] = neg_head
                    last += len(neg_head)
                if len(neg_tail) > 0:
                    batch_t[index][last : last + len(neg_tail)] = neg_tail
                    last += len(neg_tail)
            if self.neg_rel > 0:
                neg_rel = self.__rel_batch(item[0], item[1], item[2], self.neg_rel)
                batch_r[index][last : last + len(neg_rel)] = neg_rel

        batch_h = batch_h.transpose()
        batch_t = batch_t.transpose()
        batch_r = batch_r.transpose()

        batch_data["batch_h"] = batch_h.squeeze()
        batch_data["batch_t"] = batch_t.squeeze()
        batch_data["batch_r"] = batch_r.squeeze()
        return batch_data

    def __count_htr(self):
        self.h_of_tr = {}
        self.t_of_hr = {}
        self.r_of_ht = {}
        self.h_of_r = {}
        self.t_of_r = {}
        self.freqRel = {}
        self.lef_mean = {}
        self.rig_mean = {}

        triples = zip(self.head, self.tail, self.rel)
        for h, t, r in triples:
            if (h, r) not in self.t_of_hr:
                self.t_of_hr[(h, r)] = []
            self.t_of_hr[(h, r)].append(t)
            if (t, r) not in self.h_of_tr:
                self.h_of_tr[(t, r)] = []
            self.h_of_tr[(t, r)].append(h)
            if (h, t) not in self.r_of_ht:
                self.r_of_ht[(h, t)] = []
            self.r_of_ht[(h, t)].append(r)
            if r not in self.freqRel:
                self.freqRel[r] = 0
                self.h_of_r[r] = {}
                self.t_of_r[r] = {}
            self.freqRel[r] += 1.0
            self.h_of_r[r][h] = 1
            self.t_of_r[r][t] = 1

        for t, r in self.h_of_tr:
            self.h_of_tr[(t, r)] = np.array(list(set(self.h_of_tr[(t, r)])))
        for h, r in self.t_of_hr:
            self.t_of_hr[(h, r)] = np.array(list(set(self.t_of_hr[(h, r)])))
        for h, t in self.r_of_ht:
            self.r_of_ht[(h, t)] = np.array(list(set(self.r_of_ht[(h, t)])))
        for r in range(self.rel_total):
            if r in self.h_of_r:
                self.h_of_r[r] = np.array(list(self.h_of_r[r].keys()))
                self.t_of_r[r] = np.array(list(self.t_of_r[r].keys()))
                self.lef_mean[r] = self.freqRel[r] / len(self.h_of_r[r])
                self.rig_mean[r] = self.freqRel[r] / len(self.t_of_r[r])
            else:
                self.h_of_r[r] = np.array([])
                self.t_of_r[r] = np.array([])
                self.lef_mean[r] = 1.0
                self.rig_mean[r] = 1.0

    def __corrupt_head(self, t, r, num_max=1):
        tmp = np.random.randint(0, self.ent_total, size=num_max)
        if not self.filter_flag:
            return tmp
        key = (t, r)
        if key not in self.h_of_tr:
            return tmp
        mask = np.in1d(tmp, self.h_of_tr[key], assume_unique=True, invert=True)
        neg = tmp[mask]
        return neg

    def __corrupt_tail(self, h, r, num_max=1):
        tmp = np.random.randint(0, self.ent_total, size=num_max)
        if not self.filter_flag:
            return tmp
        key = (h, r)
        if key not in self.t_of_hr:
            return tmp
        mask = np.in1d(tmp, self.t_of_hr[key], assume_unique=True, invert=True)
        neg = tmp[mask]
        return neg

    def __corrupt_rel(self, h, t, num_max=1):
        tmp = np.random.randint(0, self.rel_total, size=num_max)
        if not self.filter_flag:
            return tmp
        key = (h, t)
        if key not in self.r_of_ht:
            return tmp
        mask = np.in1d(tmp, self.r_of_ht[key], assume_unique=True, invert=True)
        neg = tmp[mask]
        return neg

    def __normal_batch(self, h, t, r, neg_size):
        neg_size_h = 0
        neg_size_t = 0
        prob = (
            self.rig_mean[r] / (self.rig_mean[r] + self.lef_mean[r])
            if self.bern_flag
            else 0.5
        )
        for i in range(neg_size):
            if random.random() < prob:
                neg_size_h += 1
            else:
                neg_size_t += 1

        neg_list_h = []
        neg_cur_size = 0
        while neg_cur_size < neg_size_h:
            neg_tmp_h = self.__corrupt_head(
                t, r, num_max=(neg_size_h - neg_cur_size) * 3 + 2
            )
            neg_list_h.append(neg_tmp_h)
            neg_cur_size += len(neg_tmp_h)
        if neg_list_h != []:
            neg_list_h = np.concatenate(neg_list_h)

        neg_list_t = []
        neg_cur_size = 0
        while neg_cur_size < neg_size_t:
            neg_tmp_t = self.__corrupt_tail(
                h, r, num_max=(neg_size_t - neg_cur_size) * 3 + 2
            )
            neg_list_t.append(neg_tmp_t)
            neg_cur_size += len(neg_tmp_t)
        if neg_list_t != []:
            neg_list_t = np.concatenate(neg_list_t)

        return neg_list_h[:neg_size_h], neg_list_t[:neg_size_t]

    def __rel_batch(self, h, t, r, neg_size):
        neg_list = []
        neg_cur_size = 0
        while neg_cur_size < neg_size:
            neg_tmp = self.__corrupt_rel(
                h, t, num_max=(neg_size - neg_cur_size) * 3 + 2
            )
            neg_list.append(neg_tmp)
            neg_cur_size += len(neg_tmp)
        return np.concatenate(neg_list)[:neg_size]

    def get_ent_tot(self):
        return self.ent_total

    def get_rel_tot(self):
        return self.rel_total

    def get_tri_tot(self):
        return self.tri_total


class TransE(nn.Module):
    def __init__(
        self, ent_tot, rel_tot, dim=100, p_norm=1, norm_flag=True, margin=None
    ):
        super(TransE, self).__init__()
        self.dim = dim
        self.margin = margin
        self.norm_flag = norm_flag
        self.p_norm = p_norm
        self.ent_tot = ent_tot
        self.rel_tot = rel_tot

        self.ent_embeddings = nn.Embedding(self.ent_tot, self.dim)
        self.rel_embeddings = nn.Embedding(self.rel_tot, self.dim)

        nn.init.xavier_uniform_(self.ent_embeddings.weight.data)
        nn.init.xavier_uniform_(self.rel_embeddings.weight.data)

        if margin is not None:
            self.margin = nn.Parameter(torch.Tensor([margin]))
            self.margin.requires_grad = False
            self.margin_flag = True
        else:
            self.margin_flag = False

    def _calc(self, h, t, r):
        if self.norm_flag:
            h = F.normalize(h, p=2, dim=-1)
            t = F.normalize(t, p=2, dim=-1)
            r = F.normalize(r, p=2, dim=-1)
        score = torch.norm(h + r - t, p=self.p_norm, dim=-1)
        return score

    def forward(self, batch_h, batch_t, batch_r):
        h = self.ent_embeddings(batch_h)
        t = self.ent_embeddings(batch_t)
        r = self.rel_embeddings(batch_r)
        score = self._calc(h, t, r)
        return score

    def loss(self, pos_score, neg_score):
        if self.margin_flag:
            loss = F.relu(self.margin + pos_score - neg_score)
        else:
            loss = F.relu(pos_score - neg_score)
        return loss.mean()


def load_data_once(in_path):
    tri_file = in_path + "triple2id.txt"
    ent_file = in_path + "entity2id.txt"
    rel_file = in_path + "relation2id.txt"

    with open(ent_file, "r") as f:
        ent_total = int(f.readline())
    with open(rel_file, "r") as f:
        rel_total = int(f.readline())

    head, tail, rel = [], [], []
    with open(tri_file, "r") as f:
        triples_total = int(f.readline())
        for _ in range(triples_total):
            h, t, r = f.readline().strip().split()
            head.append(int(h))
            tail.append(int(t))
            rel.append(int(r))

    print(
        f"Data Loaded: {ent_total} entities, {rel_total} relations, {len(head)} triples."
    )

    dataset = PyTorchTrainDataset(
        np.array(head),
        np.array(tail),
        np.array(rel),
        ent_total,
        rel_total,
        sampling_mode="normal",
        bern_flag=False,
        filter_flag=True,
        neg_ent=1,
        neg_rel=0,
    )
    return dataset


def train_model(
    dataset,
    p_norm,
    margin,
    hidden_size=50,
    epochs=100,
    learning_rate=0.001,
    weight_decay=0.0,
    batch_size=4096,
    use_gpu=True,
    verbose=True,
    neg_ent=1,
):
    train_dataloader = DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        collate_fn=dataset.collate_fn,
        drop_last=True,
        pin_memory=True,
    )

    model = TransE(
        ent_tot=dataset.get_ent_tot(),
        rel_tot=dataset.get_rel_tot(),
        dim=hidden_size,
        p_norm=p_norm,
        norm_flag=True,
        margin=margin,
    )

    optimizer = optim.AdamW(
        model.parameters(), lr=learning_rate, weight_decay=weight_decay
    )
    device = torch.device("cuda" if use_gpu and torch.cuda.is_available() else "cpu")
    model.to(device)

    loss_history = []
    num_neg = neg_ent

    for epoch in range(epochs):
        ep_loss = 0.0
        num_batches = 0

        for data in train_dataloader:
            optimizer.zero_grad()
            batch_h = torch.tensor(data["batch_h"], dtype=torch.long, device=device)
            batch_t = torch.tensor(data["batch_t"], dtype=torch.long, device=device)
            batch_r = torch.tensor(data["batch_r"], dtype=torch.long, device=device)

            score = model(batch_h, batch_t, batch_r)

            batch_real_size = batch_h.size(0) // (1 + num_neg)
            pos_score = score[:batch_real_size]
            neg_score_all = score[batch_real_size:]

            if num_neg > 1:
                neg_score_all = neg_score_all.view(num_neg, batch_real_size).mean(dim=0)

            loss = model.loss(pos_score, neg_score_all)
            loss.backward()
            optimizer.step()

            ep_loss += loss.item()
            num_batches += 1

        avg_loss = ep_loss / max(num_batches, 1)
        loss_history.append(avg_loss)

        if verbose and (epoch + 1) % 50 == 0:
            print(f"Epoch {epoch + 1}/{epochs} | Loss: {avg_loss:.4f}")

        if np.isnan(avg_loss) or avg_loss > 10000:
            return model, [float("inf")]

    return model, loss_history


def save_embeddings(model, entity_file, relation_file):
    print("Saving embeddings...")
    with open(entity_file, "w") as f:
        enb = model.ent_embeddings.weight.data.cpu().numpy()
        for i in enb:
            f.write("\t".join([str(x) for x in i]) + "\n")
    with open(relation_file, "w") as f:
        rnb = model.rel_embeddings.weight.data.cpu().numpy()
        for i in rnb:
            f.write("\t".join([str(x) for x in i]) + "\n")
    print(f"Embeddings saved to {os.path.dirname(entity_file)}")


def load_id_mappings(in_path):
    entity2id, id2entity, relation2id, id2relation = {}, {}, {}, {}
    with open(in_path + "entity2id.txt", "r") as f:
        f.readline()
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                name, idx = parts[0], int(parts[1])
                entity2id[name] = idx
                id2entity[idx] = name
    with open(in_path + "relation2id.txt", "r") as f:
        f.readline()
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                name, idx = parts[0], int(parts[1])
                relation2id[name] = idx
                id2relation[idx] = name
    return entity2id, id2entity, relation2id, id2relation


def objective(trial, dataset):
    p_norm = trial.suggest_categorical("p_norm", [1, 2])
    margin = trial.suggest_float("margin", 0.5, 8.0, step=0.5)
    lr = trial.suggest_categorical("lr", [1e-4, 5e-4, 1e-3, 5e-3])
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-4, log=True)

    model, loss_history = train_model(
        dataset=dataset,
        p_norm=p_norm,
        margin=margin,
        hidden_size=50,
        epochs=30,
        learning_rate=lr,
        weight_decay=weight_decay,
        batch_size=4096,
        use_gpu=True,
        verbose=False,
    )
    return loss_history[-1]


def parameter_analysis(dataset):
    print("\n" + "=" * 60)
    print("Parameter Analysis (Fixed LR=1e-3, WD=1e-5)")
    print("=" * 60)
    p_norms = [1, 2]
    margins = [2.0, 4.0, 6.0]
    results = {}
    for p_norm in p_norms:
        for margin in margins:
            print(f"\nTraining with p_norm={p_norm}, margin={margin}")
            model, loss_history = train_model(
                dataset=dataset,
                p_norm=p_norm,
                margin=margin,
                hidden_size=50,
                epochs=200,
                learning_rate=1e-3,  # Default analysis LR for AdamW
                weight_decay=1e-5,
                batch_size=4096,
                use_gpu=True,
                verbose=False,
            )
            results[(p_norm, margin)] = {
                "final_loss": loss_history[-1],
                "loss_history": loss_history,
            }
            print(f"  Final loss: {loss_history[-1]:.4f}")
    return results


def plot_parameter_analysis(results, out_path):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax1 = axes[0]
    colors = plt.cm.tab10(np.linspace(0, 1, len(results)))
    for idx, ((p_norm, margin), data) in enumerate(results.items()):
        ax1.plot(
            data["loss_history"],
            label=f"p={p_norm}, m={margin}",
            color=colors[idx],
            alpha=0.7,
        )
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.set_title("Training Loss Curves")
    ax1.legend(fontsize=8)
    ax1.grid(True, alpha=0.3)

    ax2 = axes[1]
    margins = sorted(set(m for (_, m) in results.keys()))
    x = np.arange(len(margins))
    l1_losses = [results.get((1, m), {"final_loss": 0})["final_loss"] for m in margins]
    l2_losses = [results.get((2, m), {"final_loss": 0})["final_loss"] for m in margins]
    ax2.bar(x - 0.35 / 2, l1_losses, 0.35, label="L1 norm", color="steelblue")
    ax2.bar(x + 0.35 / 2, l2_losses, 0.35, label="L2 norm", color="coral")
    ax2.set_xlabel("Margin")
    ax2.set_ylabel("Final Loss")
    ax2.set_title("Final Loss Comparison")
    ax2.set_xticks(x)
    ax2.set_xticklabels([str(m) for m in margins])
    ax2.legend()
    ax2.grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    plt.savefig(os.path.join(out_path, "parameter_analysis.png"), dpi=150)
    print(f"\nParameter analysis plot saved to {out_path}")


def plot_optuna_results(study, out_path):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax1 = axes[0]
    trials = [t for t in study.trials if t.state == TrialState.COMPLETE]
    ax1.plot(
        [t.number for t in trials], [t.value for t in trials], "o-", color="steelblue"
    )
    ax1.axhline(y=study.best_value, color="red", linestyle="--", label="Best")
    ax1.set_xlabel("Trial")
    ax1.set_ylabel("Loss")
    ax1.set_title("Optuna History")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2 = axes[1]
    # Scatter plot of Margin vs Loss, colored by Learning Rate (since p_norm is binary)
    scatter = ax2.scatter(
        [t.params["margin"] for t in trials],
        [t.value for t in trials],
        c=[np.log10(t.params["lr"]) for t in trials],  # Log scale for LR color
        cmap="coolwarm",
        s=100,
        alpha=0.7,
    )
    plt.colorbar(scatter, ax=ax2, label="Log10(LR)")
    ax2.set_xlabel("Margin")
    ax2.set_ylabel("Loss")
    ax2.set_title("Param Exploration (Color=LR)")
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(out_path, "optuna_optimization.png"), dpi=150)
    print(f"\nOptuna plot saved to {out_path}")


def main():
    in_path = "./data/"
    out_path = "./output/"
    if not os.path.exists(out_path):
        os.makedirs(out_path)
    if not os.path.exists(in_path):
        print(f"Error: Data directory '{in_path}' not found.")
        return

    # 1. LOAD DATA ONCE
    print("=" * 60 + "\nInitializing Dataset\n" + "=" * 60)
    dataset = load_data_once(in_path)
    entity2id, id2entity, relation2id, id2relation = load_id_mappings(in_path)

    # 2. OPTUNA
    print("\n" + "=" * 60 + "\nSTEP 1: Optuna Optimization (30 Trials)\n" + "=" * 60)
    study = optuna.create_study(
        direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED)
    )
    study.optimize(
        lambda trial: objective(trial, dataset), n_trials=30, show_progress_bar=True
    )

    print(
        f"\nBest loss: {study.best_trial.value:.4f} | Params: {study.best_trial.params}"
    )
    plt.figure()
    plot_optuna_results(study, out_path)

    # 3. TRAIN FINAL MODEL
    print("\n" + "=" * 60 + "\nSTEP 2: Training Final Model\n" + "=" * 60)
    best_params = study.best_trial.params
    final_model, loss_history = train_model(
        dataset=dataset,
        p_norm=best_params["p_norm"],
        margin=best_params["margin"],
        hidden_size=50,
        epochs=1000,
        learning_rate=best_params.get("lr", 0.001),
        weight_decay=best_params.get("weight_decay", 1e-5),
        batch_size=4096,
        use_gpu=True,
        verbose=True,
    )
    save_embeddings(
        final_model,
        os.path.join(out_path, "entity2vec.txt"),
        os.path.join(out_path, "relation2vec.txt"),
    )

    # 4. CUSTOM QUERY (Customized for Q30, P36, Q49)
    print("\n" + "=" * 60 + "\nSTEP 3: Query Examples\n" + "=" * 60)

    def find_rank(target_id, all_scores, top_k_indices):
        if target_id in top_k_indices:
            return (top_k_indices == target_id).nonzero(as_tuple=True)[
                0
            ].item() + 1, True
        full_sort = torch.argsort(all_scores, descending=False)
        return (full_sort == target_id).nonzero(as_tuple=True)[0].item() + 1, False

    if "Q30" in entity2id and "P36" in relation2id:
        # --- Query 1: Head(Q30) + Rel(P36) => Tail? (Target: Q61) ---
        h_name, r_name, t_target_name = "Q30", "P36", "Q61"
        print(
            f"\n[Query 1] Head: {h_name} + Rel: {r_name} -> Find Tail (Target: {t_target_name})"
        )

        final_model.eval()
        with torch.no_grad():
            dev = next(final_model.parameters()).device
            h_idx = torch.tensor(entity2id[h_name]).to(dev)
            r_idx = torch.tensor(relation2id[r_name]).to(dev)

            h_vec = final_model.ent_embeddings(h_idx)
            r_vec = final_model.rel_embeddings(r_idx)
            if final_model.norm_flag:
                h_vec = F.normalize(h_vec, p=2, dim=-1)
                r_vec = F.normalize(r_vec, p=2, dim=-1)

            target = h_vec + r_vec
            all_ent = final_model.ent_embeddings.weight.data
            if final_model.norm_flag:
                all_ent = F.normalize(all_ent, p=2, dim=-1)

            dists = torch.norm(all_ent - target, p=final_model.p_norm, dim=-1)
            top10_val, top10_idx = torch.topk(dists, 10, largest=False)

            print("Top 10 Predictions:")
            for i in range(10):
                idx = top10_idx[i].item()
                print(
                    f" {i + 1}. {id2entity.get(idx, str(idx))} (ID:{idx}, Dist:{top10_val[i].item():.4f})"
                )

            if t_target_name in entity2id:
                t_tgt_id = entity2id[t_target_name]
                rank, is_top = find_rank(t_tgt_id, dists, top10_idx)
                print(
                    f"\nTarget {t_target_name}: Rank {rank} (Dist: {dists[t_tgt_id].item():.4f})"
                )
            else:
                print(f"\nTarget {t_target_name} not in dataset.")

        # --- Query 2: Head(Q30) + Tail(Q49) => Rel? (Target: P30) ---
        t_name, r_target_name = "Q49", "P30"
        if t_name in entity2id:
            print(
                f"\n[Query 2] Head: {h_name} + Tail: {t_name} -> Find Rel (Target: {r_target_name})"
            )
            with torch.no_grad():
                t_idx = torch.tensor(entity2id[t_name]).to(dev)
                t_vec = final_model.ent_embeddings(t_idx)
                if final_model.norm_flag:
                    t_vec = F.normalize(t_vec, p=2, dim=-1)

                target_r = t_vec - h_vec
                all_rel = final_model.rel_embeddings.weight.data
                if final_model.norm_flag:
                    all_rel = F.normalize(all_rel, p=2, dim=-1)

                dists_r = torch.norm(all_rel - target_r, p=final_model.p_norm, dim=-1)
                top10_r_val, top10_r_idx = torch.topk(dists_r, 10, largest=False)

                print("Top 10 Predictions:")
                for i in range(10):
                    idx = top10_r_idx[i].item()
                    print(
                        f" {i + 1}. {id2relation.get(idx, str(idx))} (ID:{idx}, Dist:{top10_r_val[i].item():.4f})"
                    )

                if r_target_name in relation2id:
                    r_tgt_id = relation2id[r_target_name]
                    rank, is_top = find_rank(r_tgt_id, dists_r, top10_r_idx)
                    print(
                        f"\nTarget {r_target_name}: Rank {rank} (Dist: {dists_r[r_tgt_id].item():.4f})"
                    )
                else:
                    print(f"\nTarget {r_target_name} not in dataset.")
    else:
        print("\nSkipping Custom Query: Q30/P36 not found in dataset.")

    # 5. ANALYSIS
    print("\n" + "=" * 60 + "\nSTEP 4: Parameter Analysis\n" + "=" * 60)
    analysis_results = parameter_analysis(dataset)
    plt.figure()
    plot_parameter_analysis(analysis_results, out_path)
    print(f"\nComplete! Check '{out_path}' for results.")


if __name__ == "__main__":
    main()